In [1]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
from scvi.external import MRVI
from plotnine import *
import plotly.express as px
import plotly.graph_objects as go

/home/melrie/miniforge3/envs/compare/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata = sc.read_h5ad("/home/melrie/human_intestinal/data/adata_mrvi_scvi.h5ad")

In [3]:
adata

AnnData object with n_obs × n_vars = 701840 × 18293
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'donor_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'donor_disease_category', '_indices', '_scvi_sample', '_scvi_batch', '_scvi_labels'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length

In [22]:
len(adata.obs['organ_unified'].unique())
adata.obs['cell_sorting'].unique()

#, 'age_unified', 'sample_category', 'sample_retrieval', 'tissue_type', 'donor_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', '_indices', '_scvi_sample', '_scvi_batch', '_scvi_labels'

[NaN, 'Density', 'MACS', 'FACS']
Categories (3, object): ['Density', 'FACS', 'MACS']

In [ ]:
# 1. Define which embeddings you want to include
embedding_keys = ["X_umap_unintegrated","X_umap_scvi", "X_umap_mrvi_u", "X_umap_mrvi_z"] # Add any others you have in adata.obsm
metadata_cols = ["level_1_annot", "level_2_annot", "level_3_annot", "donorID_unified", "donor_disease_category"]

In [24]:


fig = go.Figure()
trace_map = [] # To keep track of which trace is which

# --- 1. CREATE TRACES FOR EVERY COMBINATION ---
for emb in embedding_keys:
    coords = adata.obsm[emb]
    for meta in metadata_cols:
        categories = adata.obs[meta].unique()
        current_set_indices = []
        
        for cat in categories:
            mask = adata.obs[meta] == cat
            fig.add_trace(
                go.Scattergl(
                    x=coords[mask, 0],
                    y=coords[mask, 1],
                    mode='markers',
                    name=str(cat),
                    marker=dict(size=4),
                    visible=False,      # Hide everything by default
                    legendgroup=str(cat),
                    text=adata.obs_names[mask],
                    hoverinfo="text+name"
                )
            )
            current_set_indices.append(len(fig.data) - 1)
        
        trace_map.append({
            "emb": emb,
            "meta": meta,
            "indices": current_set_indices
        })

# --- 2. VISIBILITY HELPER ---
def get_visibility(emb_name, meta_name):
    vis = [False] * len(fig.data)
    for entry in trace_map:
        if entry["emb"] == emb_name and entry["meta"] == meta_name:
            for idx in entry["indices"]:
                vis[idx] = True
    return vis

# --- 3. CREATE COMBO DROPDOWN BUTTONS ---
# This creates a single list of all possible views
combo_buttons = []

for emb in embedding_keys:
    for meta in metadata_cols:
        # Create a clean label for the button
        e_label = emb.replace("X_", "").upper()
        m_label = meta.replace("_", " ").title()
        button_label = f"{e_label} | {m_label}"
        
        combo_buttons.append(dict(
            method="update",
            label=button_label,
            args=[
                {"visible": get_visibility(emb, meta)},
                {"title": f"Embedding: {e_label} | Colored by: {m_label}"}
            ]
        ))

# --- 4. FINAL LAYOUT & INITIAL STATE ---
fig.update_layout(
    updatemenus=[
        dict(
            buttons=combo_buttons,
            direction="down",
            showactive=True,
            x=0.0, 
            y=1.15, 
            xanchor="left",
            yanchor="top"
        )
    ],
    template="plotly_white",
    margin=dict(l=20, r=20, t=100, b=20),
    title=f"Embedding: {embedding_keys[0].replace('X_','').upper()} | Colored by: {metadata_cols[0].replace('_',' ').title()}"
)

# CRITICAL FIX: Set the VERY FIRST VIEW to be fully visible
# Instead of just fig.data[0], we set all categories of the first combo
initial_vis = get_visibility(embedding_keys[0], metadata_cols[0])
for i, is_vis in enumerate(initial_vis):
    fig.data[i].visible = is_vis

In [25]:
fig.write_html("interactive_umap.html")
#fig.show()